In [ ]:
import requests
import os
from dotenv import load_dotenv

# Cargar API key desde archivo .env
load_dotenv()
API_KEY = os.getenv("USDA_API_KEY")
BASE_URL = "https://api.nal.usda.gov/fdc/v1"

# Verificar conexión
response = requests.get(
    f"{BASE_URL}/foods/search",
    params={
        "api_key": API_KEY,
        "query": "breakfast cereal",
        "dataType": ["Branded"],
        "pageSize": 5
    }
)
if response.status_code == 200:
    data = response.json()
    print(f"✓ Conexión exitosa")
    print(f"Total productos: {data['totalHits']}")
    for food in data['foods']:
        print(f"  - {food['description']}")
else:
    print(f"✗ Error: {response.status_code}")

In [ ]:
import pandas as pd
import time

def extraer_cereales(api_key, max_productos=15000):
    """Extrae todos los cereales de USDA"""
    todos_los_productos = []
    page = 1
    page_size = 200  # Máximo permitido por la API
    
    while len(todos_los_productos) < max_productos:
        response = requests.get(
            f"{BASE_URL}/foods/search",
            params={
                "api_key": api_key,
                "query": "cereal",
                "dataType": ["Branded"],
                "pageSize": page_size,
                "pageNumber": page
            }
        )
        
        if response.status_code != 200:
            print(f"Error en página {page}: {response.status_code}")
            break
            
        data = response.json()
        productos = data.get('foods', [])
        
        if not productos:
            break
            
        todos_los_productos.extend(productos)
        print(f"Página {page}: {len(productos)} productos (Total: {len(todos_los_productos)})")
        
        page += 1
        time.sleep(0.5)  # Pausa para no saturar la API
    
    return todos_los_productos

# Ejecutar extracción
print("Iniciando extracción...")
productos = extraer_cereales(API_KEY)
print(f"\nTotal extraído: {len(productos)} productos")

In [ ]:
# Extraer campos relevantes
datos = []
for p in productos:
    datos.append({
        'fdc_id': p.get('fdcId'),
        'description': p.get('description'),
        'brand_owner': p.get('brandOwner'),
        'serving_size': p.get('servingSize'),
        'serving_size_unit': p.get('servingSizeUnit'),
        'food_category': p.get('foodCategory'),
        # Nutrientes
        **{n['nutrientName']: n.get('value') for n in p.get('foodNutrients', [])}
    })

# Crear DataFrame y guardar
df = pd.DataFrame(datos)
df.to_csv('C:/Users/arita/OneDrive/Escritorio/MASTER BIOINFORMATICA Y BIOESTADISTICA/TFM/data/raw/cereales_usda.csv', index=False)

print(f"Guardado: {len(df)} productos")
print(f"Columnas: {len(df.columns)}")
print(df.head())

In [ ]:
# Columnas clave 
cols_nutricionales = ['Energy', 'Protein', 'Carbohydrate, by difference', 
                      'Total lipid (fat)', 'Fatty acids, total saturated',
                      'Fiber, total dietary', 'Total Sugars', 'Sodium, Na']

cols_descriptivas = ['fdc_id', 'description', 'brand_owner', 'food_category']

# Verificar disponibilidad
print("=== VARIABLES NUTRICIONALES ===")
for col in cols_nutricionales:
    pct = (df[col].notna().sum() / len(df)) * 100
    print(f"{col}: {pct:.1f}%")

# Resumen estadístico
print("\n=== ESTADÍSTICAS ===")
df[cols_nutricionales].describe()

In [ ]:
# Seleccionar columnas finales
cols_final = cols_descriptivas + cols_nutricionales
df_clean = df[cols_final].copy()

# Eliminar filas con missings en variables nutricionales
df_clean = df_clean.dropna(subset=cols_nutricionales)

print(f"Productos originales: {len(df)}")
print(f"Productos limpios: {len(df_clean)}")
print(f"Eliminados: {len(df) - len(df_clean)} ({(len(df) - len(df_clean))/len(df)*100:.1f}%)")

# Guardar dataset limpio
df_clean.to_csv('C:/Users/arita/OneDrive/Escritorio/MASTER BIOINFORMATICA Y BIOESTADISTICA/TFM/data/processed/cereales_usda.csv', index=False)
print("\nGuardado en data/processed/cereales_clean.csv")

In [ ]:
# Resumen estadístico
print("=== ESTADÍSTICAS DESCRIPTIVAS ===")
df_clean[cols_nutricionales].describe().round(2)

In [ ]:
# Rangos válidos por 100g
rangos_validos = {
    'Energy': (0, 900),           # kcal
    'Protein': (0, 100),          # g
    'Carbohydrate, by difference': (0, 100),
    'Total lipid (fat)': (0, 100),
    'Fatty acids, total saturated': (0, 100),
    'Fiber, total dietary': (0, 100),
    'Total Sugars': (0, 100),
    'Sodium, Na': (0, 10000)      # mg
}

# Filtrar outliers
df_valid = df_clean.copy()
for col, (min_val, max_val) in rangos_validos.items():
    antes = len(df_valid)
    df_valid = df_valid[(df_valid[col] >= min_val) & (df_valid[col] <= max_val)]
    eliminados = antes - len(df_valid)
    if eliminados > 0:
        print(f"{col}: {eliminados} outliers eliminados")

print(f"\nProductos finales: {len(df_valid)}")
print(f"Eliminados total: {len(df_clean) - len(df_valid)}")

# Guardar versión final
df_valid.to_csv('C:/Users/arita/OneDrive/Escritorio/MASTER BIOINFORMATICA Y BIOESTADISTICA/TFM/data/processed/cereales_usda.csv', index=False)

In [ ]:
print("=== ESTADÍSTICAS FINALES ===")
df_valid[cols_nutricionales].describe().round(2)

In [ ]:
# Resumen final
print("=== RESUMEN DATASET ===")
print(f"Productos: {len(df_valid)}")
print(f"Variables: {len(df_valid.columns)}")
print(f"\nArchivo guardado: data/processed/cereales_clean.csv")
print("\n✓ Extracción completada")

In [ ]:
# Guardar 
df_valid.to_csv('../data/processed/cereales_clean.csv', index=False)
print(f"Verificación: {os.path.exists('../data/processed/cereales_clean.csv')}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cols_nutri = ['Energy', 'Protein', 'Carbohydrate, by difference', 
              'Total lipid (fat)', 'Fatty acids, total saturated',
              'Fiber, total dietary', 'Total Sugars', 'Sodium, Na']

# Distribuciones
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, col in enumerate(cols_nutri):
    ax = axes[i//4, i%4]
    df_valid[col].hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(col[:15])
plt.tight_layout()
plt.savefig('../docs/histogramas.png')
plt.show()

In [ ]:
# Correlaciones
plt.figure(figsize=(8, 6))
corr = df_valid[cols_nutri].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlaciones entre variables nutricionales')
plt.tight_layout()
plt.savefig('../docs/correlaciones.png')
plt.show()